In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file in ["best.pt", "last.pt"]:
            print(os.path.join(root, file))

/kaggle/input/notebooks/snigdhahy3/figo-aksunet-training/figo/runs/full_aksunet/best.pt
/kaggle/input/notebooks/snigdhahy3/figo-aksunet-training/figo/runs/full_aksunet/last.pt


In [2]:
import os
import subprocess

REPO = "/kaggle/working/figo-repo"

if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "https://github.com/acm-research-f26/Radii.git", REPO],
        check=True
    )

subprocess.run(
    ["pip", "install", "-q", "monai"],
    check=True
)

print("RADII repository and MONAI are ready!")

Cloning into '/kaggle/working/figo-repo'...


RADII repository and MONAI are ready!


In [3]:
import os

ROOT = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo"

print("Dataset exists:", os.path.exists(ROOT))

for folder in ["clean", "cache_0.9mm"]:
    path = os.path.join(ROOT, folder)
    print(f"\n{folder}:")
    if os.path.isdir(path):
        files = os.listdir(path)
        print("Number of files:", len(files))
        print("First 5 files:", files[:5])
    else:
        print("Folder not found")

print("\nSplits file exists:", os.path.exists(os.path.join(ROOT, "splits.json")))

Dataset exists: True

clean:
Number of files: 600
First 5 files: ['UMD_221129_001_seg.nii', 'UMD_221129_001_t2.nii', 'UMD_221129_002_seg.nii', 'UMD_221129_002_t2.nii', 'UMD_221129_003_seg.nii']

cache_0.9mm:
Number of files: 300
First 5 files: ['UMD_221129_001.npz', 'UMD_221129_002.npz', 'UMD_221129_003.npz', 'UMD_221129_004.npz', 'UMD_221129_005.npz']

Splits file exists: True


In [4]:
import os
import gzip
import shutil

SOURCE = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/clean"
DEST = "/kaggle/working/figo/data/clean"

os.makedirs(DEST, exist_ok=True)

for filename in os.listdir(SOURCE):
    if filename.endswith(".nii"):
        source_path = os.path.join(SOURCE, filename)
        dest_path = os.path.join(DEST, filename + ".gz")

        if not os.path.exists(dest_path):
            with open(source_path, "rb") as f_in:
                with gzip.open(dest_path, "wb") as f_out:
                    shutil.copyfileobj(f_in, f_out)

files = os.listdir(DEST)

print("Compressed files:", len(files))
print("Example files:", files[:5])
print("Preparation complete!")

Compressed files: 600
Example files: ['UMD_221129_084_seg.nii.gz', 'UMD_221129_123_seg.nii.gz', 'UMD_221129_270_t2.nii.gz', 'UMD_221129_030_t2.nii.gz', 'UMD_221129_101_seg.nii.gz']
Preparation complete!


In [5]:
import os
import subprocess

REPO = "/kaggle/working/figo-repo"

CHECKPOINT = "/kaggle/input/notebooks/snigdhahy3/figo-aksunet-training/figo/runs/full_aksunet/best.pt"

DATA = "/kaggle/working/figo/data/clean"

SPLITS = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/splits.json"

PRED = "/kaggle/working/figo/preds/aksunet"

os.makedirs(PRED, exist_ok=True)

subprocess.run(
    [
        "python", "-m", "figo.infer",
        "--checkpoint", CHECKPOINT,
        "--data-dir", DATA,
        "--splits", SPLITS,
        "--subset", "val",
        "--out-dir", PRED
    ],
    cwd=REPO,
    check=True
)

print("AKS-UNet validation inference complete!")

[1/30] UMD_221129_014
[2/30] UMD_221129_016
[3/30] UMD_221129_020
[4/30] UMD_221129_045
[5/30] UMD_221129_066
[6/30] UMD_221129_074
[7/30] UMD_221129_077
[8/30] UMD_221129_092
[9/30] UMD_221129_093
[10/30] UMD_221129_131
[11/30] UMD_221129_141
[12/30] UMD_221129_147
[13/30] UMD_221129_153
[14/30] UMD_221129_164
[15/30] UMD_221129_173
[16/30] UMD_221129_183
[17/30] UMD_221129_204
[18/30] UMD_221129_205
[19/30] UMD_221129_206
[20/30] UMD_221129_210
[21/30] UMD_221129_228
[22/30] UMD_221129_231
[23/30] UMD_221129_245
[24/30] UMD_221129_250
[25/30] UMD_221129_263
[26/30] UMD_221129_266
[27/30] UMD_221129_273
[28/30] UMD_221129_281
[29/30] UMD_221129_289
[30/30] UMD_221129_296
AKS-UNet validation inference complete!


In [6]:
import os
import subprocess

REPO = "/kaggle/working/figo-repo"
DATA = "/kaggle/working/figo/data/clean"
SPLITS = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/splits.json"
PRED = "/kaggle/working/figo/preds/aksunet"
RESULTS = "/kaggle/working/figo/results"

os.makedirs(RESULTS, exist_ok=True)

subprocess.run(
    [
        "python", "-m", "figo.evaluate",
        "--pred-dir", PRED,
        "--data-dir", DATA,
        "--splits", SPLITS,
        "--subset", "val",
        "--out", f"{RESULTS}/aksunet_seg.csv"
    ],
    cwd=REPO,
    check=True
)

print("AKS-UNet segmentation evaluation complete!")

  10/30
  20/30
  30/30

30 cases from /kaggle/working/figo/preds/aksunet
  dice_wall                           0.814 +/- 0.150
  hd95_wall                          21.866 +/- 46.296
  assd_wall                           6.450 +/- 19.099
  dice_cavity                         0.680 +/- 0.228
  hd95_cavity                        16.596 +/- 39.343  (1 missed)
  assd_cavity                         2.869 +/- 4.635  (1 missed)
  dice_myoma                          0.559 +/- 0.356
  hd95_myoma                         33.591 +/- 57.776  (1 missed)
  assd_myoma                         18.946 +/- 45.277  (1 missed)
  dice_cyst                           0.319 +/- 0.366
  hd95_cavity_wall_interface         16.446 +/- 39.267  (1 missed)
  assd_cavity_wall_interface          2.796 +/- 4.586  (1 missed)
  hd95_serosa                        24.805 +/- 48.696
  assd_serosa                         7.395 +/- 19.385
  dice_uterus                         0.849 +/- 0.199
  macro_dice_wall_cavity_myoma      

In [7]:
import os
import subprocess

REPO = "/kaggle/working/figo-repo"
DATA = "/kaggle/working/figo/data/clean"
SPLITS = "/kaggle/input/datasets/snigdhahy3/clean-datasetradii/umd-figo/splits.json"
PRED = "/kaggle/working/figo/preds/aksunet"
RESULTS = "/kaggle/working/figo/results"

os.makedirs(RESULTS, exist_ok=True)

subprocess.run(
    [
        "python", "-m", "figo.figo_eval",
        "agreement",
        "--ref-dir", DATA,
        "--pred-dir", PRED,
        "--splits", SPLITS,
        "--subset", "val",
        "--out", f"{RESULTS}/aksunet_figo_agreement.csv"
    ],
    cwd=REPO,
    check=True
)

print("AKS-UNet FIGO evaluation complete!")

  n_ref_fibroids         111
  n_pred_fibroids        100
  n_matched              57
  detection_recall       0.514
  detection_precision    0.570
  exact_agreement        0.667
  coarse_agreement       0.825
  kappa_exact            0.448
  kappa_coarse           0.638
AKS-UNet FIGO evaluation complete!


In [8]:
import os
import shutil

BASE = "/kaggle/working/figo"

# ZIP 1: Predictions from all 30 validation scans
pred_zip = shutil.make_archive(
    "/kaggle/working/aksunet_predictions",
    "zip",
    root_dir=f"{BASE}/preds",
    base_dir="aksunet"
)

# ZIP 2: Segmentation and FIGO evaluation results
results_dir = "/kaggle/working/aksunet_results_val"
os.makedirs(results_dir, exist_ok=True)

for filename in ["aksunet_seg.csv", "aksunet_figo_agreement.csv"]:
    source = f"{BASE}/results/{filename}"
    if not os.path.isfile(source):
        raise FileNotFoundError(source)
    shutil.copy2(source, results_dir)

results_zip = shutil.make_archive(
    "/kaggle/working/aksunet_results_val",
    "zip",
    root_dir="/kaggle/working",
    base_dir="aksunet_results_val"
)

print("Saved:", pred_zip)
print("Saved:", results_zip)
print("All AKS-UNet evaluation files packaged successfully!")

Saved: /kaggle/working/aksunet_predictions.zip
Saved: /kaggle/working/aksunet_results_val.zip
All AKS-UNet evaluation files packaged successfully!
